In [1]:
import numpy as np
import pandas as pd
import os
import glob
import re
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import importlib.util
import sys

In [2]:
# DATA_DIR = "../cesnet-institutions-throughput/institutions/agg_1_hour_missing"
# TRUE_DATASET = "../cesnet-institutions-throughput/institutions/agg_1_hour"
BASELINE_DIR = "./baseline"
# OUTPUT_DIR = "./imputed_results"
# EVALUATION_FILE = "./evaluation_results.csv"
# PARTIAL_EVALUATION_FILE = "./resultados_parciais_avaliacao.csv"


# os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
def load_imputation_functions():
    """Carrega todas as funções de imputação dos arquivos na pasta baselines e temporal_svd_knn"""
    functions = {}
    # Lista de diretórios para procurar por funções de imputação
    directories = [BASELINE_DIR, './temporal_svd_knn']
    
    for directory in directories:
        if not os.path.exists(directory):
            print(f"Diretório {directory} não encontrado. Pulando...")
            continue
            
        baseline_files = glob.glob(os.path.join(directory, "*.py"))
        
        for file_path in baseline_files:
            try:
                # Extrair nome do módulo
                module_name = os.path.splitext(os.path.basename(file_path))[0]
                
                # Carregar módulo
                spec = importlib.util.spec_from_file_location(module_name, file_path)
                module = importlib.util.module_from_spec(spec)
                sys.modules[module_name] = module
                spec.loader.exec_module(module)
                
                # Encontrar função de imputação (assumindo que começa com "impute_")
                for attr_name in dir(module):
                    if attr_name.startswith("impute_"):
                        functions[attr_name.replace("impute_", "")] = getattr(module, attr_name)
                        print(f"Carregada função: {attr_name} de {file_path}")
                
            except Exception as e:
                print(f"Erro ao carregar {file_path}: {e}")
    
    return functions

In [4]:
def process_datasets(data_dir, output_dir):
    """Processa todos os datasets no diretório"""
    
    # Carregar funções de imputação
    imputation_functions = load_imputation_functions()
    if not imputation_functions:
        print("Nenhuma função de imputação encontrada!")
        return
    
    # Encontrar todos os arquivos com padrão de nome
    pattern = os.path.join(data_dir, "*_esmond_data_*.csv")
    data_files = glob.glob(pattern)
    
    if not data_files:
        print(f"Nenhum arquivo encontrado com o padrão: {pattern}")
        return
    
    print(f"Encontrados {len(data_files)} arquivos para processar")
    
    rx = re.compile(
    r"""^
    (?P<prefix>.+?)_esmond_data_        # bbr (ou outro prefixo)
    (?P<link>[A-Za-z]{2}-[A-Za-z]{2})_  # go-se, ac-ap, etc.
    (?P<years>\d{4}(?:-\d{4})?)         # 2025 ou 2024-2025
    \.csv$
    """,
    re.VERBOSE | re.IGNORECASE,
)
    
    for file_path in data_files:
        try:
            # Extrair informações do nome do arquivo
            filename = os.path.basename(file_path)
            match = rx.match(filename)
            
            if not match:
                print(f"Padrão de nome inválido: {filename}")
                continue
                
            protocol = match["prefix"]
            link = match["link"].lower()        

            print(f"Processando: {filename} (ID: {protocol}, Taxa: {link}%)")
            
            # Carregar dataset
            df_missing = pd.read_csv(file_path)

            df_missing = df_missing.rename(columns={"Data": "time", "Vazao": "throughput_bps"})
            
            # Verificar se a coluna throughput_bps existe
            if "throughput_bps" not in df_missing.columns:
                print(f"Coluna 'throughput_bps' não encontrada em {filename}")
                continue            
            
            # Aplicar cada método de imputação
            for method_name, impute_func in imputation_functions.items():
                try:
                    print(f"  Aplicando {method_name}...")
                    
                    if impute_func.__name__ == 'impute_throughput_svd_knn':
                        df_imputed = impute_func(
                            df_missing,
                            col="throughput_bps",
                            min_period=24,
                            max_period=1000,
                            energy=0.9,
                            k=10,
                            allow_future=True
                        )
                    else:
                        df_imputed = impute_func(df_missing.copy())
                    
                    # Salvar resultados
                    output_file = os.path.join(output_dir, f"{protocol}_{link}_{method_name}.csv")
                    df_imputed.to_csv(output_file, index=False)
                    
                    print(f"    {method_name} concluído e salvo em {output_file}")
                    
                except Exception as e:
                    print(f"    Erro ao aplicar {method_name}: {str(e)}")
                    continue
                    
        except Exception as e:
            print(f"Erro ao processar {file_path}: {str(e)}")
            continue



In [5]:
DATA_DIR = "../data/vazao-escolhidos"
OUTPUT_DIR = "results/rnp/agg-bbr-cubic"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Executar o processamento
process_datasets(DATA_DIR, OUTPUT_DIR)


Carregada função: impute_arima de ./baseline\arima.py
Carregada função: impute_kalman de ./baseline\kalman_arima.py
Carregada função: impute_knn_imputer de ./baseline\knn.py
Carregada função: impute_mice_univariate de ./baseline\mice.py
Carregada função: impute_pca de ./baseline\pca.py
Carregada função: impute_seasonal_decompose de ./baseline\seasonal_decompose.py
Carregada função: impute_throughput_svd_knn de ./temporal_svd_knn\temporal_svd_knn.py
Encontrados 30 arquivos para processar
Processando: bbr_esmond_data_ap-ac_2024-2025.csv (ID: bbr, Taxa: ap-ac%)
  Aplicando arima...
    arima concluído e salvo em results/rnp/agg-bbr-cubic\bbr_ap-ac_arima.csv
  Aplicando kalman...
    kalman concluído e salvo em results/rnp/agg-bbr-cubic\bbr_ap-ac_kalman.csv
  Aplicando knn_imputer...
    knn_imputer concluído e salvo em results/rnp/agg-bbr-cubic\bbr_ap-ac_knn_imputer.csv
  Aplicando mice_univariate...
    mice_univariate concluído e salvo em results/rnp/agg-bbr-cubic\bbr_ap-ac_mice_univari